In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sys
np.set_printoptions(threshold=sys.maxsize)

In [ ]:
from src.config import Configuration
from src.tetris import TetrisConfiguration

T_CONFIG = TetrisConfiguration(
    board_w=4,
    board_h=10,
    vanish_zone=4, # Extra rows above the visible board to capture piece spawns
)

CONFIG = Configuration(
    max_placements=50,
    max_board_size_w=30,
    max_board_size_h=10,
)

# Train

In [ ]:
from src.tetris import Board, PieceEnum, Queue, ActionEnum, ActivePiece, Tetris, RotationEnum, ROTATION_DIR

In [ ]:
from src.models import TetrisEnv

env = TetrisEnv(CONFIG, T_CONFIG)

In [ ]:
print(env.reset()[0]["boards"].shape)
print(env.reset()[0]["queue"].shape)

### Model

In [ ]:
env.observation_space["boards"].shape

In [ ]:
from src.models import TurboMino

feature_extractor = TurboMino(
    T_CONFIG,
    CONFIG,
    env.observation_space,
)

### Correct forward pass (current model)

The feature extractor returns a single tensor `(B, max_placements)` — one scalar per placement.
The env provides a `placement_mask` so the model masks out invalid/padded slots with `-1e9`.

In [ ]:
import torch

obs, _ = env.reset()
tensor_obs = {
    "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
    "queue":  torch.as_tensor(obs["queue"], dtype=torch.float32).unsqueeze(0),
    "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
}

values = feature_extractor(tensor_obs)
print("Output shape:", values.shape)                # (1, 50)
print("Mask:", tensor_obs["placement_mask"][0])
print("Valid placements:", tensor_obs["placement_mask"].sum().item())
print("Placement values:\n", values)
print("Best placement:", values.argmax().item())

In [50]:
import time 
import tqdm

total_iters = 4*5

print('warming up...')
for i in tqdm.tqdm(range(total_iters*1000)):
    obs, _ = env.reset()
    tensor_obs = {
        "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
        "queue":  torch.as_tensor(obs["queue"], dtype=torch.float32).unsqueeze(0),
        "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
    }

    values = feature_extractor(tensor_obs)


t1 = time.time()
for i in range(total_iters):
    obs, _ = env.reset()
    tensor_obs = {
        "boards": torch.as_tensor(obs["boards"], dtype=torch.float32).unsqueeze(0),
        "queue":  torch.as_tensor(obs["queue"], dtype=torch.float32).unsqueeze(0),
        "placement_mask": torch.as_tensor(obs["placement_mask"], dtype=torch.bool).unsqueeze(0),
    }

    values = feature_extractor(tensor_obs)

t2 = time.time()

print(f"Average inference time over {total_iters} iterations: {(t2 - t1) / total_iters:.4f} seconds")

warming up...


100%|██████████| 20000/20000 [00:43<00:00, 461.87it/s]

Average inference time over 20 iterations: 0.0021 seconds
